# Auto Loader Pipeline: CSV to Delta

## Overview

This notebook demonstrates **Databricks Auto Loader** for incrementally ingesting CSV files into Delta Lake format. Auto Loader (`cloudFiles`) automatically discovers and processes new files as they arrive in cloud storage, making it ideal for building reliable data pipelines.

### What You'll Learn
* How to configure Auto Loader for CSV file ingestion
* Schema inference and evolution handling
* Writing streaming data to Delta Lake
* Checkpoint-based exactly-once processing

### Use Case
Perfect for scenarios where:
* CSV files are continuously dropped into cloud storage
* You need incremental processing (only new files)
* Schema may evolve over time
* Exactly-once processing guarantees are required

### Prerequisites
* Unity Catalog enabled workspace
* Write access to `databricksmayank.bronze` catalog
* CSV files ready to upload to the volume

## Configuration Reference

### Storage Paths
| Path Type | Location |
| --- | --- |
| **Raw CSV Files** | `/Volumes/databricksmayank/bronze/autovol/raw/` |
| **Delta Output** | `/Volumes/databricksmayank/bronze/autovol/destination/data/` |
| **Checkpoint** | `/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/` |

### Auto Loader Options Explained

**`cloudFiles.format`**: `"csv"`
* Specifies the input file format
* Supports: csv, json, parquet, avro, orc, text, binaryFile

**`cloudFiles.schemaLocation`**: Checkpoint directory
* Stores the inferred schema for consistency across runs
* Enables schema evolution tracking

**`cloudFiles.schemaEvolutionMode`**: `"rescue"`
* **rescue**: Captures unexpected columns in `_rescued_data` (recommended)
* **addNewColumns**: Automatically adds new columns to schema
* **failOnNewColumns**: Fails the stream on schema changes
* **none**: No schema evolution handling

### Write Stream Options

**`outputMode`**: `"append"`
* Adds only new records to the Delta table
* Other modes: complete, update

**`trigger`**: `once=True`
* **Batch mode**: Processes all available files then stops
* Ideal for scheduled jobs
* Alternative: `.trigger(processingTime='1 minute')` for continuous streaming

**`checkpointLocation`**
* Ensures exactly-once processing semantics
* Tracks processed files and stream state
* Critical for fault tolerance

## Step 1: Create Unity Catalog Volume

Create a volume in Unity Catalog to store both raw CSV files and processed Delta data. Volumes provide a unified namespace for managing files in cloud storage.

**Note**: Run this cell only once. If the volume already exists, you can skip this step.

## Step 2: Configure Auto Loader Stream

Set up a streaming DataFrame that monitors the raw data directory for new CSV files. Auto Loader will automatically:
* Detect new files as they arrive
* Infer the schema from CSV headers
* Track which files have been processed
* Handle schema changes gracefully

## Step 3: Write Stream to Delta Lake

Start the streaming job to process files and write results to Delta format. The `trigger(once=True)` option processes all available files once, making this ideal for scheduled batch jobs.

**Checkpoint**: Ensures exactly-once processing semantics and tracks progress.

In [0]:
%sql
create volume databricksmayank.bronze.autovol

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","csv")\
    .option("cloudFiles.schemaLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .option("cloudFiles.schemaEvolutionMode","rescue")\
    .load("/Volumes/databricksmayank/bronze/autovol/raw/")

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination/data/")

## Next Steps & Verification

### Verify the Data
```sql
-- Query the Delta table
SELECT * FROM delta.`/Volumes/databricksmayank/bronze/autovol/destination/data/`
LIMIT 10;

-- Check record count
SELECT COUNT(*) as total_records 
FROM delta.`/Volumes/databricksmayank/bronze/autovol/destination/data/`;
```

### Add New Files
To test incremental loading:
1. Upload new CSV files to `/Volumes/databricksmayank/bronze/autovol/raw/`
2. Re-run Cells 7-8 (readStream and writeStream)
3. Auto Loader will process only the new files

### Convert to Continuous Streaming
To run continuously instead of batch mode, replace:
```python
.trigger(once=True)  # Batch mode
```
with:
```python
.trigger(processingTime='1 minute')  # Continuous streaming
```

### Monitoring & Troubleshooting
* Check checkpoint location for processing state
* Review `_rescued_data` column for schema mismatches
* Monitor streaming query status with `spark.streams.active`

### Additional Resources
* [Auto Loader Documentation](https://docs.databricks.com/ingestion/auto-loader/index.html)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)

## Testing Incremental Processing

This section demonstrates Auto Loader's incremental processing capability. After uploading additional CSV files to the raw directory, re-running the write stream will process only the new files.

**What happens:**
* Auto Loader detects new files in the source directory
* Only unprocessed files are ingested (checkpoint tracks what's been processed)
* New records are appended to the existing Delta table
* Schema evolution is handled automatically via rescue mode

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination/data/")

## Verify Results: Schema Evolution

The cell below reads and displays the processed Delta table. Check for the `_rescued_data` column which captures any unexpected columns from CSV files that don't match the inferred schema.

**Schema Evolution in Action:**
* `_rescued_data` column contains JSON with unexpected columns
* Original schema remains stable
* No pipeline failures from schema changes
* You can parse `_rescued_data` downstream as needed

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/databricksmayank/bronze/autovol/destination/data/")
display(df)

## Best Practices & Production Considerations

### Performance Optimization

**File Size**
* Target 128MB - 1GB per file for optimal performance
* Too many small files impacts performance
* Use Auto Compaction for Delta tables

**Partitioning**
```python
df.writeStream.format("delta")\
  .partitionBy("date")\  # Partition by date column
  .option("checkpointLocation", checkpoint_path)\
  .start(output_path)
```

**Schema Hints** (for faster inference)
```python
.option("cloudFiles.schemaHints", "order_amount DOUBLE, order_date DATE")
```

### Monitoring & Operations

**Check Stream Status**
```python
for stream in spark.streams.active:
    print(f"Stream: {stream.name}")
    print(f"Status: {stream.status}")
    print(f"Recent progress: {stream.recentProgress}")
```

**Query Processed Files**
```python
# List processed files from checkpoint
dbutils.fs.ls("/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/sources/0/")
```

### Error Handling

**Rescue Data Analysis**
```sql
SELECT _rescued_data, COUNT(*) as rescue_count
FROM delta.`/Volumes/databricksmayank/bronze/autovol/destination/data/`
WHERE _rescued_data IS NOT NULL
GROUP BY _rescued_data;
```

**Common Issues**
* **Checkpoint conflicts**: Don't reuse checkpoint locations across different streams
* **Schema location**: Keep separate from checkpoint location
* **Volume permissions**: Ensure write access to all paths

### Production Deployment

1. **Schedule as Job**: Convert to Databricks Job for production
2. **Alerting**: Set up alerts on stream failures
3. **Cost optimization**: Use `trigger(availableNow=True)` for cost-efficient batch processing
4. **Data quality**: Add validation checks before writing
5. **Backfill**: For reprocessing, clear checkpoint and use new location

### Additional Resources

* [Auto Loader Documentation](https://docs.databricks.com/ingestion/auto-loader/index.html)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)
* [Structured Streaming Guide](https://docs.databricks.com/structured-streaming/index.html)

## Demo: Second Incremental Load

This demonstrates Auto Loader processing a second batch of files.

**Steps:**
1. Upload `orders_batch2.csv` to `/Volumes/databricksmayank/bronze/autovol/raw/`
2. Run the cell below to trigger processing
3. Auto Loader will detect and process only the new file

**What to observe:**
* Checkpoint tracks that batch1 was already processed
* Only new records from batch2 are appended
* Total record count increases
* Additional partition files created in destination

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination/data/")

## Verify: Incremental Load Results

After executing the write stream above, verify that:

**Checkpoint Behavior:**
* Check `/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/` 
* New metadata files track the processed batch2 file

**Data Files:**
* Check `/Volumes/databricksmayank/bronze/autovol/destination/data/`
* Additional Parquet partition files created
* Original batch1 data remains unchanged

**Read the combined data** in the cell below to confirm both batches are present.

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/databricksmayank/bronze/autovol/destination/data/")
display(df)

## Demo: Schema Evolution with Rescue Mode

This demonstrates Auto Loader's schema evolution handling when new columns appear in CSV files.

**Steps:**
1. Upload `orders_batch3_evolved.csv` to `/Volumes/databricksmayank/bronze/autovol/raw/`
   * This file contains **new columns**: `discount` and `payment_method`
2. Run the cell below to trigger processing

**What to observe:**
* Pipeline does NOT fail despite schema mismatch
* New columns are captured in `_rescued_data` column as JSON
* Original table schema remains stable
* Data quality is preserved for downstream processing

**Why rescue mode?**
* Prevents pipeline failures from schema changes
* Allows schema validation and cleansing downstream
* Maintains data lineage for unexpected fields

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination/data/")

## Verify: Rescued Data from Schema Evolution

After executing the write stream above, check the Delta table for rescued data.

**What you'll see:**
* Rows from `orders_batch3_evolved.csv` have data in `_rescued_data` column
* The `_rescued_data` contains JSON with the unexpected columns:
  ```json
  {"discount":"45.81", "payment_method":"UPI", "_file_path":"..."}
  ```
* Rows from batch1 and batch2 have `null` in `_rescued_data` (no schema mismatch)

**Next:** See cells 25-28 for examples of parsing `_rescued_data` JSON to extract fields.

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/databricksmayank/bronze/autovol/destination/data/")
display(df)

## Working with Rescued Data

This section demonstrates how to extract and work with data from the `_rescued_data` column.

### Approach
Since `_rescued_data` is stored as a JSON string, you need to:
1. Parse the JSON using `from_json()` with a schema
2. Extract individual fields from the parsed struct

### Examples
* **Cell 25**: ❌ Incorrect approach (kept for learning) - shows common error when trying to use dot notation on string type
* **Cell 27**: ✅ Correct approach - parse JSON first, then extract fields
* **Cell 28**: Display summary showing extracted values

### Production Recommendation
For production pipelines, consider:
* Defining explicit schemas upfront to avoid rescue mode
* Using schema hints: `.option("cloudFiles.schemaHints", "discount DOUBLE, payment_method STRING")`
* Setting `schemaEvolutionMode` to `addNewColumns` if you control the upstream schema

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn("rescued_1", col("_rescued_data.discount"))
display(df)

### Step 1: Import Required Functions

Import PySpark SQL functions and types needed for JSON parsing.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Step 2: Parse JSON and Extract Fields (Correct Approach)

This cell demonstrates the correct way to extract data from `_rescued_data`:

1. **Reload DataFrame**: Start fresh from Delta table
2. **Define schema**: Create a StructType matching the JSON structure
3. **Parse JSON**: Use `from_json()` to convert string to struct
4. **Extract fields**: Use dot notation on the struct (not the string)

The result includes new columns: `rescued_discount` and `rescued_payment_method`.

In [0]:
# Reload DataFrame fresh from Delta
df = spark.read.format("delta").load("/Volumes/databricksmayank/bronze/autovol/destination/data/")

# Define schema for rescued data
rescued_schema = StructType()\
    .add("discount", StringType())\
    .add("payment_method", StringType())

# Parse the JSON string in _rescued_data column
df = df.withColumn("rescued_struct", from_json(col("_rescued_data"), rescued_schema))

# Extract individual fields from the parsed struct
df = df.withColumn("rescued_discount", col("rescued_struct.discount"))
df = df.withColumn("rescued_payment_method", col("rescued_struct.payment_method"))

# Display the result
display(df)

### Step 3: Display Extracted Data

Display a focused view showing:
* Original order fields (`order_id`, `customer_id`, `order_amount`)
* Raw `_rescued_data` JSON string
* Extracted `rescued_discount` and `rescued_payment_method` values

This demonstrates successful extraction of the schema-evolved columns.

In [0]:
# Show only relevant columns to verify rescued data extraction
df_summary = df.select(
    "order_id",
    "customer_id", 
    "order_amount",
    "_rescued_data",
    "rescued_discount",
    "rescued_payment_method"
)

display(df_summary)

## Demo: Schema Evolution with addNewColumns Mode

This section demonstrates an alternative schema evolution strategy: **`addNewColumns`** mode.

### Difference from Rescue Mode

| Mode | Behavior | Use Case |
| --- | --- | --- |
| **rescue** | Captures new columns in `_rescued_data` as JSON | Unknown/untrusted schemas |
| **addNewColumns** | Automatically adds new columns to table schema | Controlled schema evolution |

### Setup for This Demo

To run this demo cleanly:

1. **Clear old checkpoint/data** (if rerunning):
   * Delete `/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/`
   * Delete `/Volumes/databricksmayank/bronze/autovol/destination/data/`

2. **Start fresh with new paths**:
   * Use new checkpoint: `/Volumes/databricksmayank/bronze/autovol/destination2/checkpoint/`
   * Use new data path: `/Volumes/databricksmayank/bronze/autovol/destination2/data/`

3. **Upload files in order**:
   * First: `orders_batch1.csv` (base schema)
   * Then: `orders_batch3_evolved.csv` (with new columns: discount, payment_method)

**What to observe:** New columns are automatically added to the Delta table schema instead of being captured in `_rescued_data`.

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","csv")\
    .option("cloudFiles.schemaLocation","/Volumes/databricksmayank/bronze/autovol/destination2/checkpoint/")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("/Volumes/databricksmayank/bronze/autovol/raw/")

### Step 1: Configure Auto Loader with addNewColumns

Create a streaming DataFrame with `schemaEvolutionMode` set to `addNewColumns`.

**Key difference:** When new columns appear in CSV files, they will be automatically added to the table schema instead of being captured in `_rescued_data`.

### Step 2: Write Stream with mergeSchema Option

Write the streaming data to Delta with the `mergeSchema` option enabled.

**`mergeSchema: true`** allows the Delta table to accept schema changes from the incoming stream:
* New columns from `addNewColumns` mode are added to the table
* Existing columns remain unchanged
* Schema is automatically merged during write operations

**Important:** Both `cloudFiles.schemaEvolutionMode=addNewColumns` (read side) and `mergeSchema=true` (write side) must be enabled for automatic schema evolution to work.

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination2/checkpoint/")\
    .option("mergeSchema","true")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination2/data/")

### Step 3: Verify Automatic Schema Evolution

Read the Delta table to verify that new columns were automatically added to the schema.

**Expected results:**
* Table now includes `discount` and `payment_method` columns (not in `_rescued_data`)
* New columns are first-class columns in the schema
* Rows from original batch have `null` values in new columns
* Rows from evolved batch have actual values in new columns

**Compare with rescue mode:** No JSON parsing needed - new columns are directly accessible.

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/databricksmayank/bronze/autovol/destination2/data/")
display(df)

## Demo: Second Batch with addNewColumns Mode

Now upload the evolved CSV file to demonstrate schema evolution with `addNewColumns` mode.

**Steps:**
1. Upload `orders_batch3_evolved.csv` to `/Volumes/databricksmayank/bronze/autovol/raw/`
2. Run cell 41 to process the new file

**What to observe:**
* New columns (`discount`, `payment_method`) are automatically added to the table schema
* `_rescued_data` remains `null` for all rows
* No JSON parsing needed - new columns are directly accessible
* Schema evolution happens seamlessly without pipeline failure

In [0]:
# Reload streaming DataFrame
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","csv")\
    .option("cloudFiles.schemaLocation","/Volumes/databricksmayank/bronze/autovol/destination2/checkpoint/")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("/Volumes/databricksmayank/bronze/autovol/raw/")

# Write stream to Delta
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination2/checkpoint/")\
    .option("mergeSchema","true")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination2/data/")

### Verify: Schema Evolved with New Columns

Read the Delta table to see the complete results after processing both batches.

**Expected schema:**
* Original columns: `order_id`, `customer_id`, `order_date`, `order_amount`, `order_status`
* New columns: `discount`, `payment_method`
* `_rescued_data`: Present but `null` for all rows

**Key observation:** Unlike rescue mode, the new columns are part of the main schema, not captured in JSON.

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/databricksmayank/bronze/autovol/destination2/data/")
display(df)

## Summary

Congratulations! You've successfully completed the Auto Loader tutorial. 

### What You Accomplished

✅ **Set up Auto Loader** streaming ingestion from CSV to Delta  
✅ **Demonstrated incremental processing** with multiple file batches  
✅ **Handled schema evolution** using rescue mode  
✅ **Extracted rescued data** from JSON strings into structured columns  
✅ **Learned checkpoint-based** exactly-once processing  

### Key Takeaways

1. **Auto Loader simplifies file ingestion** - Automatically discovers new files and tracks what's been processed
2. **Schema evolution is manageable** - Rescue mode prevents pipeline failures from schema changes
3. **Checkpoints ensure reliability** - Exactly-once processing semantics with automatic state management
4. **Delta Lake provides ACID guarantees** - Reliable, performant storage for your data

### Next Steps

**For Development:**
* Experiment with different `schemaEvolutionMode` options
* Try other file formats (JSON, Parquet, Avro)
* Add data quality checks and transformations
* Implement error handling and monitoring

**For Production:**
* Convert this notebook to a Databricks Job for scheduling
* Set up alerting on stream failures
* Implement data quality validation before writing
* Consider using `trigger(availableNow=True)` for cost-efficient batch processing
* Add retry logic and dead-letter queues for failed records

**Additional Learning:**
* [Auto Loader Advanced Configuration](https://docs.databricks.com/ingestion/auto-loader/options.html)
* [Delta Lake Optimization Techniques](https://docs.databricks.com/delta/optimizations/index.html)
* [Building Production Data Pipelines](https://docs.databricks.com/workflows/index.html)

---

**Questions or Issues?**  
Refer to the [Databricks Community](https://community.databricks.com/) or [Documentation](https://docs.databricks.com/)